In [ ]:
import numpy as np
import glob as glob

import matplotlib.pyplot as plt

In [ ]:
def clean_dictionary(dd, keep_keys=None):
    ''' Takes in dictionary, outputs dictionary with only the keys in keep_keys'''
    out = {}
    for key in dd.keys():
        if key in keep_keys:
            out[key] = dd[key]
    return out

In [ ]:
fbase = '/home/puehlengt/appletree/notebooks/3_params'
prefix = 'testsims_3params'
needs_cleaning = False
num_sims = 20000
fnames = glob.glob(f'{fbase}/{prefix}_{num_sims:.0f}sims*.npy')

# open one file to get structure
bag = np.load(fnames[0], allow_pickle=True).item()

if needs_cleaning:
    # find which parameters are floated
    floated_params = []
    for key, val in bag['param_bag'][1].items():
        if isinstance(val, np.float64): # if it's a floated parameter
            floated_params.append(key)

    bag['param_bag'] = [clean_dictionary(_, keep_keys=floated_params) for _ in bag['param_bag']]

In [ ]:
for _f in fnames[1:]:
    tmp_dict = np.load(_f, allow_pickle=True).item()

    if needs_cleaning:
        tmp_dict['param_bag'] = [clean_dictionary(_, keep_keys=floated_params) for _ in tmp_dict['param_bag']]

    # appending 
    for k in bag.keys():
        bag[k].extend(tmp_dict[k])

In [ ]:
assert len(bag['events_bag']) == num_sims*len(fnames), 'Missing some simulations.'

In [ ]:
#save_on = True
save_on = False
save_name = f'{fbase}/harvested_{prefix}.npy'

print(f'Saving to {save_name}...')
if save_on:
    np.save(save_name, bag)

In [ ]:
raise

In [ ]:
n = [np.shape(tmp)[1] for tmp in bag['events_bag']]
rate = [tmp['ambe_nr_rate'] for tmp in bag['param_bag']]
plt.hist(rate, bins=100, histtype='step', density=True, label='rate')
plt.hist(n, bins=100, histtype='step', density=True, label='n')
plt.title(f'{len(bag["events_bag"]):.1e} pseudo-experiments')
plt.legend()